# 01. Smoke test: sentiment BiGRU
Перший запуск на сервері: середовище, інференс, короткий train-smoke. Ноутбук лише викликає скрипти з `src/`.

In [ ]:
import json, os, platform, subprocess, sys
from pathlib import Path
import pandas as pd, torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("ROOT:", ROOT)
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| cpus", os.cpu_count())

def run(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode: print(r.stderr[-3000:]); raise SystemExit(r.returncode)

## 1. Наявність файлів

In [ ]:
for p in ["models/sentiment_rnn.pt","configs/config.yaml","data/processed/labeled.csv","data/samples/sample_reviews.csv"]:
    f = ROOT/p; print(("OK  " if f.exists() else "MISSING"), p, f"{f.stat().st_size/1e6:.1f} MB" if f.exists() else "")

## 2. Predict smoke
Вердикт за правилом: `p < 0.15` negative, `p >= 0.60` positive, між ними neutral.

In [ ]:
run("src/predict.py", "doesn't have any flaws", "i will never come to this bank again",
    "great sound but the battery is bad", "works fine", "the speaker broke after a week")
pd.read_csv("runs/smoke_predict/predictions.csv")[["text","p_positive","entropy","label","oov_share"]]

In [ ]:
df = pd.read_csv("runs/smoke_predict/predictions.csv")
ax = df.set_index("text")["p_positive"].plot.barh(figsize=(7,3), title="p(positive)")
ax.axvline(0.15, ls="--", c="gray"); ax.axvline(0.60, ls="--", c="gray")

## 3. Train smoke
Одна епоха на малій вибірці, лише щоб переконатися, що пайплайн проходить. Метрики тут ні про що не свідчать.

In [ ]:
run("src/train.py", "--smoke")
print(json.dumps(json.load(open("runs/smoke_train/metrics.json")), indent=2, ensure_ascii=False))

## 4. Результати
`runs/smoke_predict/predictions.csv` і `runs/smoke_train/{metrics.json,history.json,sentiment_rnn.pt}`.

In [ ]:
for p in sorted((ROOT/"runs").rglob("*")):
    if p.is_file(): print(p.relative_to(ROOT), f"{p.stat().st_size/1e3:.0f} KB")